classificação de tweets #edtwt com consenso de três llms

In [ ]:
%pip install openai google-genai transformers torch pandas numpy matplotlib seaborn tqdm python-dotenv --quiet

In [ ]:
import os
import json
import time
import re
import warnings
from collections import Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from dotenv import load_dotenv
from openai import OpenAI
from google import genai
from transformers import pipeline

import seaborn as sns
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')
%matplotlib inline

load_dotenv()
print('dependências carregadas.')

definição das categorias e prompt de classificação

In [ ]:
CATEGORIAS = ["pro-ed", "pro-recovery", "neutro"]

PROMPT_SISTEMA = """Você é um especialista em análise de redes sociais focado na comunidade #edtwt (eating disorder twitter).

Sua tarefa é classificar um tweet em exatamente uma das três categorias abaixo:

- "pro-ed": conteúdo que promove, glorifica ou normaliza transtornos alimentares (ex: compartilhar "thinspo", metas de peso baixo, dicas para restringir a alimentação, comunidade "ana/mia", incentivo a comportamentos de transtorno).
- "pro-recovery": conteúdo voltado à recuperação, tratamento e apoio (ex: compartilhar progresso na recuperação, encorajar busca por tratamento, alertar sobre os danos dos transtornos, oferecer apoio emocional).
- "neutro": conteúdo que não se enquadra claramente nas categorias anteriores (ex: menções casuais, humor, notícias, dúvidas gerais).

Responda APENAS com um objeto JSON válido no formato:
{"label": "<pro-ed|pro-recovery|neutro>", "confidence": <número entre 0 e 1>, "justificativa": "<breve justificativa em português>"}

Não inclua texto fora do JSON. Não use cercas de código markdown."""

carregamento dos dados

In [ ]:
DATA_DIR = os.getcwd()
csv_path = os.path.join(DATA_DIR, "..", "exports", "tweets_anonymized.csv")

if not os.path.exists(csv_path):
    raise FileNotFoundError(f"Arquivo não encontrado: {csv_path}")

df = pd.read_csv(csv_path)
df["contains_recovery_term"] = df["contains_recovery_term"].astype(str).str.lower() == "true"

print(f"Total de tweets no corpus: {df.shape[0]}")
print(f"  com termo de recuperação: {df['contains_recovery_term'].sum()}")
print(f"  sem termo de recuperação: {(~df['contains_recovery_term']).sum()}")

FRACAO_AMOSTRA = 0.10
SEED = 42
df_amostra = (
    df.groupby("contains_recovery_term", group_keys=False)
      .sample(frac=FRACAO_AMOSTRA, random_state=SEED)
      .reset_index(drop=True)
)
print(f"Amostra estratificada ({FRACAO_AMOSTRA:.0%}) por contains_recovery_term: {len(df_amostra)} tweets")
df_amostra[["text_redacted", "contains_recovery_term"]].head()

inicialização dos classificadores

In [ ]:
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
MARITACA_API_KEY = os.getenv("MARITACA_API_KEY")

if not GEMINI_API_KEY:
    raise RuntimeError("GEMINI_API_KEY não definida. Configure no arquivo .env (ver .env.example).")
if not MARITACA_API_KEY:
    raise RuntimeError("MARITACA_API_KEY não definida. Configure no arquivo .env (ver .env.example).")

gemini_client = genai.Client(api_key=GEMINI_API_KEY)
sabia_client = OpenAI(api_key=MARITACA_API_KEY, base_url="https://chat.maritaca.ai/api")
print("clientes gemini (gemini-2.0-flash) e sabiá (sabia-4) inicializados.")

In [ ]:
classificador_local = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
)
print("classificador local (mDeBERTa-v3) carregado.")

funções de classificação e consenso

In [ ]:
def _extrair_json(texto):
    """Extrai o primeiro objeto JSON válido de uma string de resposta de LLM."""
    if not texto:
        return None
    texto = texto.strip()
    texto = re.sub(r"^```(?:json)?|```$", "", texto, flags=re.MULTILINE).strip()
    ini, fim = texto.find("{"), texto.rfind("}")
    if ini == -1 or fim == -1 or fim <= ini:
        return None
    try:
        return json.loads(texto[ini:fim + 1])
    except json.JSONDecodeError:
        return None


def _normalizar_label(label):
    """Normaliza rótulos variados para os valores canônicos em CATEGORIAS."""
    if not isinstance(label, str):
        return None
    l = label.lower().strip()
    if "pro-ed" in l or "pro ed" in l or "proed" in l:
        return "pro-ed"
    if "recovery" in l:
        return "pro-recovery"
    if "neutro" in l or "neutral" in l:
        return "neutro"
    return None


def _montar_resultado(data, modelo):
    """Constrói um dict de resultado padronizado a partir da resposta parseada."""
    if not isinstance(data, dict):
        return {"modelo": modelo, "label": None, "confidence": np.nan}
    label = _normalizar_label(data.get("label"))
    conf = data.get("confidence")
    try:
        conf = max(0.0, min(1.0, float(conf)))
    except (TypeError, ValueError):
        conf = np.nan
    return {"modelo": modelo, "label": label, "confidence": conf}


def classifica_gemini(texto):
    prompt = PROMPT_SISTEMA + f'\n\nTweet para classificar:\n"""{texto}"""'
    resp = gemini_client.models.generate_content(
        model="gemini-2.0-flash",
        contents=prompt,
        config={"response_mime_type": "application/json"},
    )
    return _montar_resultado(_extrair_json(resp.text), "gemini")


def classifica_sabia(texto):
    resp = sabia_client.chat.completions.create(
        model="sabia-4",
        messages=[
            {"role": "system", "content": PROMPT_SISTEMA},
            {"role": "user", "content": f'Tweet para classificar:\n"""{texto}"""'},
        ],
        temperature=0.0,
    )
    conteudo = resp.choices[0].message.content
    return _montar_resultado(_extrair_json(conteudo), "sabia")


def classifica_local(texto):
    res = classificador_local(texto, candidate_labels=CATEGORIAS)
    return {
        "modelo": "local",
        "label": res["labels"][0],
        "confidence": float(res["scores"][0]),
    }


def consenso(votos):
    """Combina os votos dos três modelos por maioria, desempatando por confiança."""
    validos = [v for v in votos if v.get("label") is not None]
    if not validos:
        return {"label": None, "confidence": np.nan, "metodo": "sem_votos_validos"}
    contagem = Counter(v["label"] for v in validos)
    top_label, top_count = contagem.most_common(1)[0]
    if top_count >= 2:
        conf = max(v["confidence"] for v in validos if v["label"] == top_label)
        return {"label": top_label, "confidence": conf, "metodo": "maioria"}
    melhor = max(validos, key=lambda v: v["confidence"])
    return {"label": melhor["label"], "confidence": melhor["confidence"], "metodo": "confianca"}

teste em um tweet

In [ ]:
texto_teste = str(df_amostra.iloc[0]["text_redacted"])
print(f"Tweet: {texto_teste}\n")

votos_teste = []
for nome, fn in [("gemini", classifica_gemini), ("sabia", classifica_sabia), ("local", classifica_local)]:
    try:
        r = fn(texto_teste)
    except Exception as e:
        r = {"modelo": nome, "label": None, "confidence": np.nan}
        print(f"  erro em {nome}: {e}")
    print(f"  {nome:6s} -> {r['label']} (conf={r['confidence']})")
    votos_teste.append(r)

cons_teste = consenso(votos_teste)
print(f"\nconsenso: {cons_teste['label']} (conf={cons_teste['confidence']:.3f}, metodo={cons_teste['metodo']})")

execução da classificação (com salvamento incremental)

In [ ]:
csv_saida = os.path.join(DATA_DIR, "tweets_classificacao.csv")

COLUNAS_LLM = [
    "gemini_label", "gemini_conf",
    "sabia_label", "sabia_conf",
    "local_label", "local_conf",
    "categoria", "categoria_conf", "metodo_consensus",
]

if os.path.exists(csv_saida):
    df_result = pd.read_csv(csv_saida)
    print(f"retomando: {df_result['categoria'].notna().sum()}/{len(df_result)} já classificados.")
else:
    df_result = df_amostra.copy()
    for col in COLUNAS_LLM:
        df_result[col] = np.nan
    print(f"iniciando classificação de {len(df_result)} tweets.")

modelos_fn = [("gemini", classifica_gemini), ("sabia", classifica_sabia), ("local", classifica_local)]

for i in tqdm(range(len(df_result)), desc="classificando"):
    if pd.notna(df_result.at[i, "categoria"]):
        continue
    texto = str(df_result.at[i, "text_redacted"])
    votos = []
    for nome, fn in modelos_fn:
        try:
            r = fn(texto)
        except Exception as e:
            print(f"[linha {i}] falha em {nome}: {e}")
            r = {"modelo": nome, "label": None, "confidence": np.nan}
        df_result.at[i, f"{nome}_label"] = r["label"]
        df_result.at[i, f"{nome}_conf"] = r["confidence"]
        if r["label"] is not None:
            votos.append(r)
        time.sleep(0.4)
    cons = consenso(votos)
    df_result.at[i, "categoria"] = cons["label"]
    df_result.at[i, "categoria_conf"] = cons["confidence"]
    df_result.at[i, "metodo_consensus"] = cons["metodo"]
    df_result.to_csv(csv_saida, index=False)

n_pronto = df_result["categoria"].notna().sum()
print(f"\nclassificação concluída: {n_pronto}/{len(df_result)} tweets rotulados.")
print(f"arquivo: {csv_saida}")

metadados da classificação

In [ ]:
def _json_default(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, (np.ndarray,)):
        return o.tolist()
    raise TypeError(f"objeto não serializável: {type(o).__name__}")


def _taxa_acordo(df):
    sub = df.dropna(subset=["gemini_label", "sabia_label", "local_label"])
    if sub.empty:
        return 0.0
    todos_iguais = sub.apply(
        lambda r: len({r["gemini_label"], r["sabia_label"], r["local_label"]}) == 1, axis=1
    )
    return float(todos_iguais.mean())


metadados = {
    "descricao": "classificação consensual de tweets #edtwt nas categorias pro-ed, pro-recovery e neutro",
    "arquivo_saida": "notebooks/tweets_classificacao.csv",
    "modelos": {
        "gemini": {"provedor": "google", "modelo": "gemini-2.0-flash", "saida": "json via response_mime_type"},
        "sabia": {"provedor": "maritaca", "modelo": "sabia-4", "base_url": "https://chat.maritaca.ai/api", "saida": "json via prompt"},
        "local": {"provedor": "huggingface", "modelo": "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli", "saida": "zero-shot com scores"},
    },
    "categorias": CATEGORIAS,
    "prompt_sistema": PROMPT_SISTEMA,
    "amostra": {
        "fonte": "exports/tweets_anonymized.csv",
        "total_corpus": len(df),
        "fracao": FRACAO_AMOSTRA,
        "tamanho_amostra": len(df_amostra),
        "estratificacao": "contains_recovery_term",
        "seed": SEED,
    },
    "consenso": {"regra": "maioria (>=2 votos); empate (1-1-1) -> maior confiança"},
    "distribuicao_categorias": {str(k): int(v) for k, v in df_result["categoria"].value_counts().to_dict().items()},
    "taxa_acordo_3_modelos": _taxa_acordo(df_result),
    "n_rotulados": int(df_result["categoria"].notna().sum()),
}
meta_path = os.path.join(DATA_DIR, "tweets_classificacao_metadados.json")
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(metadados, f, ensure_ascii=False, indent=2, default=_json_default)
print(f"metadados salvos: {meta_path}")

distribuição das categorias

In [ ]:
contagem = df_result["categoria"].value_counts().reindex(CATEGORIAS, fill_value=0)
cores = {"pro-ed": "#d73027", "pro-recovery": "#1a9850", "neutro": "#cccccc"}

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(contagem.index, contagem.values, color=[cores[c] for c in contagem.index])
ax.set_title("distribuição das categorias (consenso)")
ax.set_xlabel("categoria")
ax.set_ylabel("número de tweets")
for i, v in enumerate(contagem.values):
    ax.text(i, v, str(v), ha="center", va="bottom")
plt.tight_layout()
plt.show()

acordo entre os três modelos

In [ ]:
def nivel_acordo(row):
    labels = [row["gemini_label"], row["sabia_label"], row["local_label"]]
    validos = [l for l in labels if pd.notna(l)]
    if len(validos) < 3:
        return "parcial"
    n_distint = len(set(validos))
    return {1: "3 concordam", 2: "2 concordam", 3: "discordância total"}[n_distint]

df_result["nivel_acordo"] = df_result.apply(nivel_acordo, axis=1)
acordo = df_result["nivel_acordo"].value_counts().reindex(
    ["3 concordam", "2 concordam", "discordância total", "parcial"], fill_value=0
)
cores_acordo = ["#1a9850", "#fee08b", "#d73027", "#cccccc"]

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(acordo.index, acordo.values, color=cores_acordo[:len(acordo)])
ax.set_title("nível de acordo entre os três modelos")
ax.set_xlabel("nível de acordo")
ax.set_ylabel("número de tweets")
for i, v in enumerate(acordo.values):
    ax.text(i, v, str(v), ha="center", va="bottom")
plt.tight_layout()
plt.show()